# Phase 6 — Evaluation

Test-set accuracy / macro-F1 / weighted-F1 / confusion matrix / per-class precision-recall, reported **overall** and broken down **per DS1/DS2/DS3** — the EDA's KS tests confirmed real domain shift across sub-datasets, so a pooled number alone can hide a model that only learned DS1.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # notebooks/ -> project/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from project.config import load_config, resolve_path

cfg = load_config()


## Baseline (SlowFast)

In [2]:
from project.datasets.feature_dataset import CachedEmbeddingDataset
from project.models.baseline_mlp import SlowFastBaselineMLP
from project.models.encoders import get_device
from project.training.utils import load_checkpoint
from project.evaluation.evaluate import evaluate_model

device = get_device(cfg['encoders']['device_preference'])
slowfast_dir = resolve_path(cfg, 'paths.features_dir') / 'slowfast'
test_csv = resolve_path(cfg, 'paths.test_csv')
test_ds = CachedEmbeddingDataset(test_csv, [(slowfast_dir, 'parent_video_id')])

baseline_model = SlowFastBaselineMLP(input_dim=2304, hidden_dim=cfg['training']['baseline_hidden_dim'],
                                      num_classes=len(cfg['emotion_to_id']), dropout=cfg['training']['baseline_dropout'])
load_checkpoint(baseline_model, resolve_path(cfg, 'paths.checkpoints_dir') / 'baseline_slowfast_mlp_best.pt')
baseline_results = evaluate_model(baseline_model, test_ds, test_csv, cfg, device,
                                   'baseline_slowfast', resolve_path(cfg, 'paths.results_dir'))


[baseline_slowfast] OVERALL  n=478  acc=0.5063  macro_f1=0.4787  weighted_f1=0.5027
[baseline_slowfast]  DS1     n=388  acc=0.5490  macro_f1=0.5045  weighted_f1=0.5428
[baseline_slowfast]  DS2     n=59  acc=0.3390  macro_f1=0.3378  weighted_f1=0.3800
[baseline_slowfast]  DS3     n=31  acc=0.2903  macro_f1=0.2586  weighted_f1=0.3193
[baseline_slowfast] metrics -> /Users/rafiabhista/Documents/code/apple/c1_audio/emomv_2/project/evaluation/results/baseline_slowfast_metrics.json


## Fusion (VideoMAE + CLIP)

In [3]:
from project.models.encoders import FrozenCLIPEncoder, FrozenVideoMAEEncoder
from project.models.fusion_classifier import FusionClassifier

videomae_dir = resolve_path(cfg, 'paths.videomae_features_dir')
clip_dir = resolve_path(cfg, 'paths.clip_features_dir')
test_ds_fusion = CachedEmbeddingDataset(test_csv, [(videomae_dir, 'video_id'), (clip_dir, 'video_id')])

fusion_model = FusionClassifier(videomae_dim=FrozenVideoMAEEncoder.EMBED_DIM, clip_dim=FrozenCLIPEncoder.EMBED_DIM,
                                 hidden_dim=cfg['training']['fusion_hidden_dim'], num_classes=len(cfg['emotion_to_id']),
                                 dropout=cfg['training']['fusion_dropout'])
load_checkpoint(fusion_model, resolve_path(cfg, 'paths.checkpoints_dir') / 'fusion_classifier_best.pt')
fusion_results = evaluate_model(fusion_model, test_ds_fusion, test_csv, cfg, device,
                                 'fusion_videomae_clip', resolve_path(cfg, 'paths.results_dir'))


[fusion_videomae_clip] OVERALL  n=478  acc=0.8013  macro_f1=0.7940  weighted_f1=0.8033
[fusion_videomae_clip]  DS1     n=388  acc=0.8660  macro_f1=0.8596  weighted_f1=0.8671
[fusion_videomae_clip]  DS2     n=59  acc=0.5254  macro_f1=0.4732  weighted_f1=0.5447
[fusion_videomae_clip]  DS3     n=31  acc=0.5161  macro_f1=0.3660  weighted_f1=0.5503
[fusion_videomae_clip] metrics -> /Users/rafiabhista/Documents/code/apple/c1_audio/emomv_2/project/evaluation/results/fusion_videomae_clip_metrics.json


## Baseline vs. fusion — quick comparison

In [4]:
import pandas as pd

comparison = pd.DataFrame({
    'baseline_slowfast': {k: v for k, v in baseline_results['overall'].items() if k in ('accuracy','macro_f1','weighted_f1')},
    'fusion_videomae_clip': {k: v for k, v in fusion_results['overall'].items() if k in ('accuracy','macro_f1','weighted_f1')},
})
comparison

,baseline_slowfast,fusion_videomae_clip
accuracy,0.506276,0.801255
macro_f1,0.478681,0.793967
weighted_f1,0.502673,0.803327
